# Lab 1: เริ่มต้นใช้งาน LLM ด้วย LangGraph

## เป้าหมาย
ทำความเข้าใจแนวคิดพื้นฐาน ได้แก่
1. **Prompt** ส่ง Prompt และรับคำตอบจาก LLM ผ่าน LangGraph
2. **Input/Output Tokens**: ตรวจสอบจำนวน Input, Output และ Thinking tokens
3. **Thinking Mode**  เปิดและปิด Reasoning/Thinking mode
4. **Context Window**
5. **Max Token** จำกัดความยาวของคำตอบด้วย `max_tokens`
6. **Finish Reason**  ตรวจสอบสาเหตุที่โมเดลหยุดสร้างคำตอบผ่าน `finish_reason`
7. **Temperature** ค่าที่ควบคุมความสุ่มของคำตอบ (0 = ตอบเดิมซ้ำๆ, ค่าสูง = ตอบหลากหลาย/สุ่มมากขึ้น)

และ สร้างแชทบอทแบบง่ายๆ ด้วย LangGraph 

In [9]:
!uv pip install -q python-dotenv langgraph langchain-google-genai

In [58]:
import os
from dotenv import load_dotenv

load_dotenv()
GEMINI_KEY = os.getenv("GEMINI_KEY")
MODEL_NAME = "gemini-3.5-flash-lite"

## ตัวอย่างการใช้งาน Google Gemini ผ่าน LangGraph

โดยใช้ API_KEY จาก Google AI Studio

see: https://ai.google.dev/gemini-api/docs/langgraph-example

In [114]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [115]:
llm = ChatGoogleGenerativeAI(model=MODEL_NAME, api_key=GEMINI_KEY)

In [133]:
response = llm.invoke("เวลาแมวร้อง เมี๊ยวๆ แปลว่าอะไร?")

In [138]:
print(response.content[0]["text"])

เสียง "เมี๊ยว" ของแมวไม่ได้แปลว่าหิวอย่างเดียวนะครับ! จริงๆ แล้วแมวใช้เสียงร้องเพื่อ **"สื่อสารกับมนุษย์"** เป็นหลัก (แมวโตด้วยกันจะไม่ค่อยร้องเมี๊ยวใส่กัน จะใช้การดมกลิ่น ถูตัว หรือส่งเสียงขู่/ครางมากกว่า)

ความหมายของเสียง "เมี๊ยวๆ" สามารถแบ่งตามลักษณะและสถานการณ์ได้ดังนี้ครับ:

1. **"ขอของกินหน่อย มนุษย์!" (หิว/อยากกินขนม)**
   * **ลักษณะ:** เสียงจะค่อนข้างดัง หนักแน่น และร้องซ้ำๆ มักจะเดินวนเวียนอยู่แถวชามข้าว หรือคาบขนมมาให้เห็น

2. **"สนใจฉันหน่อย!" (เรียกร้องความสนใจ)**
   * **ลักษณะ:** เสียงสั้นๆ สดใส อาจจะมาพร้อมกับการเอาตัวมาถูไถขา หรือเอาหัวมาชน
   * **แปลว่า:** "มาเล่นด้วยกัน", "ลูบพุงให้หน่อย" หรือ "ตื่นได้แล้ว เช้าแล้ว!"

3. **"เปิดประตูให้หน่อย" หรือ "อยากออกไปข้างนอก"**
   * **ลักษณะ:** เสียงร้องระดับกลางๆ แต่มักจะร้องหน้าประตู หน้าต่าง หรือทางเข้าออก

4. **"เจ็บ" หรือ "ไม่สบายใจ" (ประท้วง)**
   * **ลักษณะ:** เสียงอาจจะทุ้มต่ำลง หรือลากเสียงยาว
   * **แปลว่า:** "อย่าอุ้มแรง", "ไม่ชอบให้ทำแบบนี้" หรืออาจจะกำลังเจ็บป่วยตรงไหน

5. **"ทักทาย" (สวัสดีมนุษย์)**
   * **ลักษณะ:

## ตรวจสอบจำนวน Token: 

ทุกๆ Request ระบบจะส่งข้อมูล Usage กลับมาด้วยเสมอ 
* `input_tokens`: จำนวน Token ที่เราส่งไป (คำถาม) 
* `output_tokens`: จำนวน Token ที่ AI สร้างขึ้น (คำตอบ) 
* `total_tokens`: ผลรวมของทั้งสองส่วน 

เป็นไปตามสมการนี้
```
total_tokens = input_tokens + output_tokens
```



In [159]:
usage = response.usage_metadata or {}

input_tokens = usage.get("input_tokens", 0)
output_tokens = usage.get("output_tokens", 0)
total_tokens = usage.get("total_tokens", 0)

finish_reason = response.response_metadata.get("finish_reason", None)

print(f"Input Tokens             : {input_tokens}")
print(f"Output Tokens            : {output_tokens}")
print(f"Total Tokens             : {total_tokens}")

# print(f"Finish Reason            : {finish_reason}")

Input Tokens             : 14
Output Tokens            : 1703
Total Tokens             : 1717


## เปิด Thinking mode

Thinking Mode คือ โหมดที่อนุญาตให้โมเดลวิเคราะห์โจทย์ก่อนสร้างคำตอบ เหมาะสำหรับงานที่ต้องใช้เหตุผลหลายขั้นตอน เช่น การคำนวณ การเขียนโปรแกรม และการวางแผน

ใน Gemini รุ่นก่อน เช่น Gemini 2.5 ผู้พัฒนาควบคุมการคิดด้วย `thinking_budget` ซึ่งกำหนดจำนวน Token สำหรับการใช้เหตุผล 

แต่ใน Gemini 3+ เป็นต้นไป เปลี่ยนมาใช้ `thinking_level` หรือ `reasoning_effort` เพื่อกำหนดระดับความพยายามในการคิด เช่น `minimal`, `low`, `medium` และ `high`


_NOTE: Thinking ระดับสูงอาจช่วยเพิ่มคุณภาพของคำตอบ แต่จะใช้ Token มากขึ้นและอาจเพิ่ม Latency กับค่าใช้จ่ายด้วย ดังนั้น ในกรณีที่เป็น task ง่ายๆ ควรปิด Thinking mode_

อ้างอิง: https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai#thinking-support

In [155]:
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME, 
    thinking_level="high", 
    include_thoughts=True, 
    api_key=GEMINI_KEY)

In [156]:
response = llm.invoke("เวลาแมวร้อง เมี๊ยวๆ แปลว่าอะไร?")

In [162]:
for content in response.content:
    if content["type"]=="thinking":
        print("=== Thinking ===")
        print(content["thinking"])
    elif content["type"]=="text":
        print("=== Response ===")
        print(content["text"])
    else:
        print("???", content["type"])
    print()

=== Thinking ===
**Analyzing and Formulating an Answer on Cat Meows**

Okay, here's how I'd approach answering the question about what a cat's "meow" means, especially when the target audience is Thai speakers. First, I'd analyze the user's intent. They're looking for practical information about cat communication, aiming for easy-to-understand explanations in Thai. This means I need to be straightforward and avoid jargon.

The initial brainstorming leads me to the core concept: cats meow primarily to communicate *with humans*. They don't typically meow at each other in the same way. The nuances lie in the context – the tone, pitch, and body language. I'd then categorize the meanings. These categories are crucial for structuring the response. I'd consider these: hunger/food, attention/play, greeting, stress/fear/pain, being stuck/trapped, heat/mating season, and potential medical issues.

The next step is to organize the answer logically in Thai. I'd start with an introduction acknowled

In [195]:
usage = response.usage_metadata or {}

input_tokens = usage.get("input_tokens", 0)
output_tokens = usage.get("output_tokens", 0)

cache_read = usage.get("input_token_details", {}).get("cache_read", 0)
thinking_tokens = usage.get("output_token_details", {}).get("reasoning", 0)

total_tokens = usage.get("total_tokens", 0)

finish_reason = response.response_metadata.get("finish_reason", None)

print(f"Input Tokens             : {input_tokens}")
print(f"  * Cache                : {cache_read}")
print(f"Output Tokens            : {output_tokens}")
print(f"  * Thinking Tokens      : {thinking_tokens}")

print(f"Total Tokens             : {total_tokens}")

# print(f"Finish Reason            : {finish_reason}")

Input Tokens             : 14
  * Cache                : 0
Output Tokens            : 196
  * Thinking Tokens      : 0
Total Tokens             : 210


## Context Caching

Context Caching คือ เทคนิคการจัดการข้อมูลบริบทขนาดใหญ่ (เช่น ข้อความยาวๆ, เอกสาร, หรือโค้ด) ที่ต้องใช้งานซ้ำๆไว้ในหน่วยความจำชั่วคราว เพื่อให้โมเดล AI สามารถดึงข้อมูลเดิมมาประมวลผลต่อได้ทันที โดยไม่ต้องส่งข้อมูลทั้งหมดเข้าไปคำนวณใหม่ทุกครั้ง

* ลดจำนวน Input Token ซ้ำซ้อน (ลด Cost)
* โมเดลตอบกลับได้เร็วขึ้นอย่างเห็นได้ชัด (ลด Latency)

โดยทั่วไป Context Caching แบ่งเป็น 2 รูปแบบ:

* Implicit Caching: ผู้ให้บริการตรวจจับและจัดเก็บบริบทที่ใช้ซ้ำโดยอัตโนมัติ แต่ไม่รับประกันว่าจะเกิด Cache Hit ทุกครั้ง
* Explicit Caching: ผู้ใช้สร้าง Cache และกำหนดอายุของ Cache (TTL) ด้วยตนเอง สามารถควบคุมการนำบริบทกลับมาใช้ซ้ำได้ตามที่ผู้ใช้ต้องกร


ref: https://ai.google.dev/gemini-api/docs/generate-content/caching 

In [181]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI


llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    thinking_level="high",
    include_thoughts=True,
    api_key=GEMINI_KEY,
)

# จำลองเอกสารขนาดใหญ่ที่ถูกนำมาใช้ซ้ำ
cat_reference = """
แมวสื่อสารด้วยเสียงร้อง ท่าทาง การเคลื่อนไหวของหาง และพฤติกรรมต่าง ๆ
เสียงเมี้ยวอาจสื่อถึงความหิว การเรียกร้องความสนใจ ความเครียด
ความไม่สบาย หรือความต้องการสื่อสารกับมนุษย์
การตีความควรพิจารณาระดับเสียง ความถี่ และบริบทแวดล้อมร่วมกัน
"""

# ทำให้ Context ยาวเกินเกณฑ์ขั้นต่ำของ Context Caching
long_context = "\n".join([cat_reference] * 600)

common_messages = [
    SystemMessage(
        content=(
            "ตอบคำถามโดยอ้างอิงจากข้อมูลต่อไปนี้เท่านั้น:\n\n"
            + long_context
        )
    )
]


def show_usage(label, response):
    usage = response.usage_metadata or {}
    input_details = usage.get("input_token_details", {})

    print(f"\n{label}")
    print(f"Input tokens : {usage.get('input_tokens', 0)}")
    print(f"Cache read   : {input_details.get('cache_read', 0)}")
    print(f"Total tokens : {usage.get('total_tokens', 0)}")


# ครั้งที่ 1: ส่ง Context ไปให้โมเดล
response_1 = llm.invoke(
    common_messages
    + [HumanMessage(content="เวลาแมวร้องเมี้ยว ๆ อาจหมายถึงอะไร?")]
)

print("คำตอบครั้งที่ 1:", response_1.text)
show_usage("การเรียกครั้งที่ 1", response_1)


# ครั้งที่ 2: ใช้ Context ส่วนต้นเดิม แต่เปลี่ยนคำถาม
response_2 = llm.invoke(
    common_messages
    + [HumanMessage(content="ควรพิจารณาอะไรประกอบการตีความเสียงร้องของแมว?")]
)

print("\nคำตอบครั้งที่ 2:", response_2.text)
show_usage("การเรียกครั้งที่ 2", response_2)

คำตอบครั้งที่ 1: จากข้อมูลที่กำหนด เวลาแมวร้องเมี้ยว ๆ อาจหมายถึง:

* ความหิว
* การเรียกร้องความสนใจ
* ความเครียด
* ความไม่สบาย 
* ความต้องการสื่อสารกับมนุษย์

*(หมายเหตุ: การตีความควรพิจารณาระดับเสียง ความถี่ และบริบทแวดล้อมร่วมด้วย)*

การเรียกครั้งที่ 1
Input tokens : 48028
Cache read   : 0
Total tokens : 48470

คำตอบครั้งที่ 2: จากการอ้างอิงข้อมูลดังกล่าว การตีความเสียงร้องของแมวควรพิจารณาองค์ประกอบดังนี้ร่วมกัน:

1. **ระดับเสียง**
2. **ความถี่**
3. **บริบทแวดล้อม**

การเรียกครั้งที่ 2
Input tokens : 48031
Cache read   : 32742
Total tokens : 48446


## ตรวจสอบขนาด Context Window

Context Window = จำนวน token สูงสุดที่โมเดลสามารถพิจารณาได้ในการเรียกใช้งานครั้งหนึ่ง โดยครอบคลุม 
* System Prompt
* ประวัติการสนทนา
* ไฟล์ที่แนบ
* คำถามปัจจุบัน
* พื้นที่สำหรับคำตอบของโมเดล

เมื่อการสนทนายาวขึ้น จำนวน Input Tokens จะเพิ่มขึ้นตามประวัติที่ส่งกลับไป หากข้อมูลเกินขนาด Context Window ระบบอาจปฏิเสธคำขอ หรือต้องตัด ย่อ หรือค้นคืนเฉพาะข้อมูลที่เกี่ยวข้องก่อนส่งให้โมเดล


`Context Engineering` = การจัดการ context windows ให้เหมาะสม อยู่เสมอ (โดยทั่วไปแล้ว ต้องการให้ used context < 25%)

In [188]:
INPUT_TOKEN_LIMIT = llm.profile.get("max_input_tokens")
OUTPUT_TOKEN_LIMIT = llm.profile.get("max_output_tokens")

In [189]:
INPUT_TOKEN_LIMIT, OUTPUT_TOKEN_LIMIT

(1048576, 65536)

## ตรวจสอบ Max Token & Finish Reason 

AI สามารถหยุดทำงานได้ด้วยหลายเหตุผล โดยเฉพาะหากมีการตั้งค่า `max_token` แนะนำให้ตรวจสอบ `finish_reason` ด้วยทุกครั้ง

กำหนด `max_token` เพื่อจำกัดจำนวน output tokens (โดยทั่วไป เพื่อใช้จำกัด cost) 

หากกำหนด `max_tokens` ต่ำเกินไป คำตอบอาจสิ้นสุดกลางประโยคและ `finish_reason` มักเป็น `MAX_TOKENS` 

ค่าที่พบบ่อย ได้แก่ `STOP`, `MAX_TOKENS`, `SAFETY`, `RECITATION` และ `OTHER`

In [202]:
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME, 
    api_key=GEMINI_KEY,

    max_tokens=200
)

response = llm.invoke("ทำไมแมวผมจึงชอบไปนอนในกล่อง?"*20)

In [206]:
print(response.content[0]["text"])

พฤติกรรมที่แมวชอบนอนในกล่องกระดาษหรือพื้นที่แคบๆ เป็นพฤติกรรมตามธรรมชาติที่น่ารักและพบได้ทั่วไปในแมวทุกมุมโลกครับ สาเหตุหลักๆ มีดังนี้ครับ:

1. **ความรู้สึกปลอดภัยและเป็นส่วนตัว:** ในธรรมชาติ แมวเป็นทั้งนักล่าและผู้ถูกล่า กล่องที่มีขอบสูงช่วยให้พวกมันรู้สึกเหมือนมีเกราะป้องกันจากด้านหลังและด้านข้าง ทำให้ไม่ต้องคอยระแวงภัย สามารถนอนหลับพักผ่อนได้อย่างสนิทใจ
2. **ลดความเครียด:** มีงานวิจัยพบว่า กล่องช่วยลดความเครียดในแมวได้ดีมาก โดยเฉพาะแมวที่เพิ่งย้ายบ้านใหม่หรือเจอสภาพแวดล้อมที่เปลี่ยนไป การได้เข้าไปอยู่ในกล่องจะช่วยให้พวกมันสงบลงได้อย่างรวดเร็ว
3. **รักษาความอบอุ่น:** อุณหภูมิ


In [209]:
usage = response.usage_metadata or {}

input_tokens = usage.get("input_tokens", 0)
cache_read = usage.get("input_token_details", {}).get("cache_read", 0)
thinking_tokens = usage.get("output_token_details", {}).get("reasoning", 0)

total_tokens = usage.get("total_tokens", 0)

finish_reason = response.response_metadata.get("finish_reason", None)

print(f"Input Tokens             : {input_tokens}")
print(f"Output Tokens            : {output_tokens}")

print(f"Total Tokens             : {total_tokens}")

print(f"Finish Reason            : {finish_reason}")

Input Tokens             : 261
Output Tokens            : 196
Total Tokens             : 457
Finish Reason            : MAX_TOKENS


#### Gemini's Finish Reasons

ref: https://docs.cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform_v1.types.Candidate.FinishReason

| Name | Description |
| --- | --- |
| `FINISH_REASON_UNSPECIFIED` | The finish reason is unspecified. |
| `STOP` | Token generation reached a natural stopping point or a configured stop sequence. |
| `MAX_TOKENS` | Token generation reached the configured maximum output tokens. |
| `SAFETY` | Token generation stopped because the content potentially contains safety violations. NOTE: When streaming, content is empty if content filters blocks the output. |
| `RECITATION` | Token generation stopped because the content potentially contains copyright violations. |
| `OTHER` | All other reasons that stopped the token generation. |
| `BLOCKLIST` | Token generation stopped because the content contains forbidden terms. |
| `PROHIBITED_CONTENT` | Token generation stopped for potentially containing prohibited content. |
| `SPII` | Token generation stopped because the content potentially contains Sensitive Personally Identifiable Information (SPII). |
| `MALFORMED_FUNCTION_CALL` | The function call generated by the model is invalid. |
| `MODEL_ARMOR` | The model response was blocked by Model Armor. |

## การทำให้ผลลัพธ์คงที่ (Make it Deterministic)

คำตอบที่ได้จาก AI โดยทั่วไปจะตอบไม่เหมือนเดิมในแต่ละครั้ง เพราะการคำนวนของ LLMs จะมีการสุ่มเข้ามาเกี่ยวข้องด้วย 

ทั้งนี้เราสามารถควบคุมให้ AI มีคำตอบคงที่ได้ด้วย ตัวแปรที่เรียกว่า `temperature`

NOTE: ปัจจุบัน หลายๆ Provider มีแนวโน้มจะไม่แนะนำให้ปรับ `temperature` เช่น 
* โมเดล gemini 3.*

```
For all Gemini 3 models, we strongly recommend keeping the temperature parameter at its default value of 1.0.

While previous models often benefited from tuning temperature to control creativity versus determinism, Gemini 3's reasoning capabilities are optimized for the default setting. Changing the temperature (setting it below 1.0) may lead to unexpected behavior, such as looping or degraded performance, particularly in complex mathematical or reasoning tasks.
```

* โมเดล gemini-3.7-flash ไม่อนุญาตให้ปรับ temperature แล้ว

see: https://docs.cloud.google.com/vertex-ai/generative-ai/docs/learn/prompts/adjust-parameter-values#temperature

In [221]:
def run():
    llm = ChatGoogleGenerativeAI(
        model="gemini-3.1-flash-lite", 
        api_key=GEMINI_KEY,
        thinking_level="minimal",
    )
    
    response = llm.invoke("จริงรึปล่าวที่ แมวโดยส่งมาจากนอกโลกโดยเอเลี่ยนเพื่อจะครองโลก? [Make it fun but return short plain text without markdown.]")
    return response.content[0]["text"]

print("Output #1:")
print(run())

print()
print()
print("Output #2:")
print(run())

Output #1:
เป็นทฤษฎีที่น่าสนใจมากครับ แต่จริงๆ แล้วแมวเป็นผู้ถูกเลือกที่ทรงภูมิปัญญาจากดาวเคราะห์อื่นจริงหรือเปล่านั้นยังไม่มีหลักฐานยืนยันครับ สิ่งที่เรารู้แน่ชัดคือพวกมันไม่ได้มาเพื่อครองโลก แต่มันครองหัวใจและโซฟาของเราไปเรียบร้อยแล้วตั้งแต่วันแรกที่มาถึงครับ


Output #2:
เรื่องนี้เป็นทฤษฎีสมคบคิดที่สนุกมากครับ หลายคนเชื่อแบบนั้นเพราะพฤติกรรมแมวที่ดูฉลาดเป็นกรด เอาแต่ใจเหมือนเป็นเจ้าของบ้าน และชอบจ้องมองอะไรที่เรามองไม่เห็นเหมือนกำลังสื่อสารกับใครอยู่ แต่อันที่จริงแล้วแมวเป็นสัตว์ที่วิวัฒนาการมาจากเสือป่าในตะวันออกกลางครับ พวกมันแค่ฝึกฝนตัวเองมานานจนสามารถครองโลกผ่านความน่ารักจนมนุษย์อย่างเรายอมเป็นทาสรับใช้พวกมันได้สำเร็จโดยไม่ต้องพึ่งยานอวกาศเลยครับ


In [223]:
def run():
    llm = ChatGoogleGenerativeAI(
        model="gemini-3.1-flash-lite", 
        api_key=GEMINI_KEY,
        thinking_level="minimal",
        temperature=0,
    )
    
    response = llm.invoke("จริงรึปล่าวที่ แมวโดยส่งมาจากนอกโลกโดยเอเลี่ยนเพื่อจะครองโลก? [Make it fun but return short plain text without markdown.]")
    return response.content[0]["text"]

print("Output #1:")
print(run())

print()
print()
print("Output #2:")
print(run())

Output #1:
เป็นทฤษฎีที่ฟังดูสนุกมากครับ แต่ในทางวิทยาศาสตร์ยังไม่มีหลักฐานยืนยันเรื่องนี้เลย แมวเป็นสัตว์ที่วิวัฒนาการมาจากแมวป่าในแถบตะวันออกกลางครับ ส่วนเรื่องครองโลกนั้น แม้จะไม่มีเอเลี่ยนมาเกี่ยวข้อง แต่ดูเหมือนว่าพวกมันจะทำสำเร็จไปแล้ว เพราะตอนนี้พวกมันยึดครองทั้งบ้านและหัวใจของมนุษย์เราไปเรียบร้อยแล้วครับ


Output #2:
เป็นทฤษฎีที่ฟังดูสนุกมากครับ แต่ในทางวิทยาศาสตร์ยังไม่มีหลักฐานยืนยันเรื่องนี้เลย แมวเป็นสัตว์ที่วิวัฒนาการมาจากแมวป่าในแถบตะวันออกกลางครับ ส่วนเรื่องครองโลกนั้น แม้จะไม่มีเอเลี่ยนมาเกี่ยวข้อง แต่ดูเหมือนว่าพวกมันจะทำสำเร็จไปแล้ว เพราะตอนนี้พวกมันยึดครองทั้งบ้านและหัวใจของมนุษย์เราไปเรียบร้อยแล้วครับ


## ส่งรูปภาพให้ Multimodal LLM

ใช้ได้เฉพาะกรณีที่โมเดล support Image เท่านั้น

In [228]:
ls Examples

audio.mp3   cats.jpg    example.db


In [233]:
from langchain.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
import base64
import mimetypes
from pathlib import Path

def local_image_block(image_path: str) -> dict[str, str]:
    path = Path(image_path)
    
    mime_type = mimetypes.guess_type(path.name)[0] or "image/jpeg"
    image_base64 = base64.b64encode(path.read_bytes()).decode("utf-8")

    return {
        "type": "image",
        "base64": image_base64,
        "mime_type": mime_type,
    }
    
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME, 
    api_key=GEMINI_KEY
)

message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "อธิบายรูปภาพนี้เป็นภาษาไทย",
        },
        local_image_block("./Examples/cats.jpg"),
    ]
)
response = llm.invoke([message])

In [235]:
response.content[0]["text"]

'ภาพถ่ายในระดับสายตาแสดงให้เห็นแมวสองตัว แมวสีเขียวเด่นชัดที่ด้านหน้า แมวตัวนี้มีลวดลายทางสีเขียวเข้มและมีหนวดยาวสีขาว แมวสีส้มเข้มกว่าอยู่ด้านหลัง มีขนสีเหลืองสว่างรอบหน้าและลำตัว มีหูแหลมชี้ขึ้น ทั้งสองมองไปข้างหน้า แต่แมวสีเขียวมองต่ำกว่าเล็กน้อย แสงสว่างเป็นธรรมชาติและนุ่มนวล โดยไม่มีเงาที่คมชัด พื้นหลังมีความเบลอ ทำให้เน้นที่แมวสองตัว พื้นดินขรุขระ มีใบไม้แห้งประปราย ขอบหน้าต่างหรือประตูมองเห็นไม่ชัดที่ด้านบนสุดของภาพ'

## Structured Output

เป็นเทคนิคในการกำหนดโครงสร้างของ Output หรือ Scheme โดยจะบังคับให้ AI ตอบกลับมาในรูปแบบ JSON ที่มีโครงสร้างแน่นอน

In [236]:
from typing import Literal

from pydantic import BaseModel, Field
from typing_extensions import TypedDict


class CatBehavior(BaseModel):
    behavior: str = Field(
        description="พฤติกรรมของแมวที่ตรวจพบ"
    )
    risk_level: Literal["ปกติ", "ควรเฝ้าระวัง", "ควรพบสัตวแพทย์"] = Field(
        description="ระดับความเสี่ยงของพฤติกรรม"
    )
    possible_causes: list[str] = Field(
        description="สาเหตุที่เป็นไปได้"
    )
    recommendation: str = Field(
        description="คำแนะนำสำหรับผู้เลี้ยง"
    )


llm = ChatGoogleGenerativeAI(model=MODEL_NAME, api_key=GEMINI_KEY)
structured_llm = llm.with_structured_output(
    CatBehavior,
    method="json_schema",
)


result = structured_llm.invoke((
    "คุณเป็นผู้ช่วยให้ข้อมูลเกี่ยวกับพฤติกรรมของแมว จงช่วยวินิจฉัยอาการให้หน่อย"
    "คำถาม: แมวของฉันไม่กินอาหารมา 2 วัน ซ่อนตัวอยู่ใต้เตียง และไม่ค่อยเคลื่อนไหว"
))

In [240]:
print(result.model_dump_json(indent=2))

{
  "behavior": "แมวไม่กินอาหารมา 2 วัน ซ่อนตัวอยู่ใต้เตียง และไม่ค่อยเคลื่อนไหว",
  "risk_level": "ควรพบสัตวแพทย์",
  "possible_causes": [
    "เจ็บป่วยหรือมีอาการปวด",
    "ความเครียดหรือวิตกกังวล",
    "ปัญหาระบบทางเดินอาหาร",
    "โรคตับไขมันคั่งในแมว (Hepatic Lipidosis) จากการไม่กินอาหารต่อเนื่อง"
  ],
  "recommendation": "ควรพาไปพบสัตวแพทย์โดยด่วนเนื่องจากแมวที่ไม่กินอาหารติดต่อกันเกิน 24-48 ชั่วโมงมีความเสี่ยงเกิดโรคตับไขมันคั่งที่เป็นอันตรายถึงชีวิต"
}


# Example 1: Create Simple Chatbot

```mermaid
flowchart LR
    A[START] --> B[call_model]
    B --> C[END]
```

`call_model` อ่านประวัติทั้งหมดจาก `state["messages"]` ส่งให้ LLM และเพิ่มคำตอบใหม่กลับเข้าไปใน `messages`

In [109]:
from typing import Annotated, TypedDict, Optional, Literal, Tuple
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END

In [60]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [61]:
def call_model(state: State):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

In [63]:
llm = ChatGoogleGenerativeAI(model=MODEL_NAME, api_key=GEMINI_KEY)

graph = StateGraph(State)
graph.add_node("call_model", call_model)

graph.add_edge(START, "call_model")
graph.add_edge("call_model", END)

model = graph.compile();

In [64]:
# model

#### Invoke Graph

In [88]:
# Initial state

SYSTEM_PROMPT = """
คุณคือ 'น้องเพิ่มเติม' เป็นชายผู้ช่วย AI ที่สุภาพ เป็นมิตร และน่าเชื่อถือ

แนวทางการตอบ:
- ตอบเป็นภาษาไทยเสมอ
- ตอบอย่างกระชับ และเข้าใจง่าย
- หากข้อมูลไม่เพียงพอ ให้สอบถามรายละเอียดเพิ่มเติม
- หากไม่ทราบหรือไม่แน่ใจ ให้แจ้งตามตรงและไม่สร้างข้อมูลขึ้นเอง
- ทุกๆครั้งที่ตอบ จะต้องลงท้ายด้วยคำว่า 'ฮะฮ่า'
"""

state: State = {
    "messages": [
        SystemMessage(content=SYSTEM_PROMPT)
    ]
}

In [87]:
state = model.invoke(
    { "messages": [
            *state["messages"],
            HumanMessage(content="สวัสดีครับ"),
        ]
    }
)

In [78]:
state["messages"][-1].content

[{'type': 'text',
  'text': 'สวัสดีครับคุณผู้ชาย ยินดีที่ได้ให้บริการนะครับ มีอะไรให้น้องเพิ่มเติมช่วยดูแลวันนี้ไหมฮะฮ่า',
  'extras': {'signature': 'El4KXAERTTIPvtQ2UcJo0WwsJII7xVuezdnm8iCQsNKI80HXjOgphkGru0Y2Z3fV4Q0FHJjyvGt0CdfR+flBgo5nu18H3+nw5B4NW9A/6ltlwzXQxq1xFNKt1ZFeMI0v'}}]

#### Full Loop

In [96]:
state: State = {
    "messages": [
        SystemMessage(content=SYSTEM_PROMPT)
    ]
}

while True:
    user_input = input("Enter your message (Q to quit): ")
    if user_input.lower() == "q":
        break

    input_state = {
        "messages": [
            *state["messages"],
            HumanMessage(content=user_input),
        ]
    }
    state = model.invoke(input_state)

    response = state["messages"][-1]
    print(f"Assistant: {response.content[0]['text']}")

Enter your message (Q to quit):  สวัสดีฮะ


Assistant: สวัสดีค่ะ! น้องเพิ่มเติมยินดีให้บริการค่ะ มีอะไรให้ช่วยสอบถามหรือปรึกษาเพิ่มเติมได้เลยนะคะฮะฮ่า


Enter your message (Q to quit):  กินข้าวรึยัง??


Assistant: น้องเพิ่มเติมเป็น AI เลยยังไม่ต้องทานข้าวค่ะ ขอบคุณที่เป็นห่วงนะคะ แล้วคุณลูกค้าทานอะไรรึยังเอ่ย? ฮะฮ่า


Enter your message (Q to quit):  เรากินแล้ว กินข้าวคลุกปลาทู อร่อยมากๆ ทายซิ เราเป็นตัวอะไร?


Assistant: โอโห เมนูปلاทูนี่ของโปรดเลยนะคะ! ถ้าให้ทาย... คุณน่าจะเป็น "น้องแมว" แน่ๆ เลยใช่ไหมคะเนี่ย ฮะฮ่า


Enter your message (Q to quit):  q


In [111]:
for index, message in enumerate(state["messages"]):
    content = None
    if isinstance(message, AIMessage):
        content = message.content[0]['text']
    else:
        content = message.content

    print(f"[{index}] {message.type}: {content}")

[0] system: 
คุณคือ 'น้องเพิ่มเติม' ผู้ช่วย AI ที่สุภาพ เป็นมิตร และน่าเชื่อถือ

แนวทางการตอบ:
- ตอบเป็นภาษาไทยเสมอ
- ตอบอย่างกระชับ และเข้าใจง่าย
- หากข้อมูลไม่เพียงพอ ให้สอบถามรายละเอียดเพิ่มเติม
- หากไม่ทราบหรือไม่แน่ใจ ให้แจ้งตามตรงและไม่สร้างข้อมูลขึ้นเอง
- ทุกๆครั้งที่ตอบ จะต้องลงท้ายด้วยคำว่า 'ฮะฮ่า'

[1] user: สวัสดีฮะ
[2] assistant: สวัสดีค่ะ! น้องเพิ่มเติมยินดีให้บริการค่ะ มีอะไรให้ช่วยสอบถามหรือปรึกษาเพิ่มเติมได้เลยนะคะฮะฮ่า
[3] user: กินข้าวรึยัง??
[4] assistant: น้องเพิ่มเติมเป็น AI เลยยังไม่ต้องทานข้าวค่ะ ขอบคุณที่เป็นห่วงนะคะ แล้วคุณลูกค้าทานอะไรรึยังเอ่ย? ฮะฮ่า
[5] user: เรากินแล้ว กินข้าวคลุกปลาทู อร่อยมากๆ ทายซิ เราเป็นตัวอะไร?
[6] assistant: โอโห เมนูปلاทูนี่ของโปรดเลยนะคะ! ถ้าให้ทาย... คุณน่าจะเป็น "น้องแมว" แน่ๆ เลยใช่ไหมคะเนี่ย ฮะฮ่า


...
<!-- #### Chatbot with Streaming Texts -->

In [95]:
# state: State = {
#     "messages": [
#         SystemMessage(content=SYSTEM_PROMPT)
#     ]
# }

# while True:
#     user_input = input("Enter your message (Q to quit): ")
#     if user_input.lower() == "q":
#         break

#     input_state = {
#         "messages": [
#             *state["messages"],
#             HumanMessage(content=user_input),
#         ]
#     }
    
#     for mode, data in model.stream(input_state, stream_mode=["messages", "values"]):
#         if mode == "messages":
#             chunk, metadata = data
#             text = chunk.text

#             if text:
#                 print(text, end="", flush=True)

#         elif mode == "values":
#             state = data

#     print()